# Modelo: de MODIS a VIIRS
**Objetivo:** con lo que ve MODIS en un pedazo del mapa durante un mes, estimar lo que habría visto VIIRS en ese mismo lugar y mes. El modelo aprende en los años en que los dos sensores operaron juntos y después se aplica a 2002–2011, cuando solo existía MODIS. Así se obtiene una serie larga en "unidades VIIRS".

```
Entradas: MODIS en la celda y el mes            Salidas: VIIRS en la misma celda y mes
  detecciones y FRP de Terra y Aqua       →       detecciones
  fuego en celdas vecinas y meses vecinos  →       FRP total
  posición del Sol, lugar y fecha
```

**Pasos**
1. Configuración
2. Tabla celda × mes (SQL)
3. Variables calculadas (feature engineering)
4. Separar años
5. Tres modelos: referencia, LightGBM y red neuronal
6. ¿Qué tan bien predicen?
7. ¿Qué entradas usa el modelo?
8. ¿Qué pasa con las órbitas corridas de MODIS?
9. Validación año por año
10. ¿Dónde se equivoca? (mapa)
11. Serie armonizada 2002–2025, con su rango de incertidumbre
12. Antes y después de armonizar

La primera vez, correr todo tarda ~25 min. La tabla y la validación se guardan en `data/modelo/`, y las siguientes veces todo tarda ~10 min.

---
## 1. Configuración
Las gráficas están en [graficas_modelo.py](graficas_modelo.py) y el estilo en [estilo.py](estilo.py), para que aquí solo quede la lógica del modelo.

In [ ]:
import sys
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import r2_score
from torch import nn

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks"))
import graficas_modelo as graf

# LightGBM y PyTorch traen cada uno su propio OpenMP y en Mac se bloquean entre sí.
# Con un solo hilo PyTorch no usa OpenMP, y la red igual entrena una época en ~5 s.
torch.set_num_threads(1)
_ = torch.manual_seed(42)

GRADO = 0.5  # tamaño de cada celda del mapa, en grados (~55 km)
ARCHIVO_TABLA = ROOT / "data" / "modelo" / f"tabla_celdas_{GRADO}.parquet"

# Base en memoria para los pasos intermedios; la base de la NASA se abre solo para leer
con = duckdb.connect()
con.execute(f"ATTACH '{ROOT / 'data' / 'db_nasa.duckdb'}' AS nasa (READ_ONLY)")

---
## 2. Tabla celda × mes
Se arma en SQL en 7 pasos. Cada paso es una **vista**: una consulta con nombre que no calcula nada hasta que se usa. Cualquier paso se puede revisar con `con.sql("SELECT * FROM nombre_de_la_vista LIMIT 5")`.

**Paso 1 · Cada detección con su celda y su hora local.** La celda sale de dividir la latitud y la longitud entre 0.5°. La hora solar local es la hora UTC más longitud ÷ 15, porque cada 15° de longitud es una hora.

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW detecciones AS
SELECT
    satellite, sensor, daynight, type, frp, acq_date, latitude,
    CAST(floor(latitude / {GRADO}) AS INT)  AS fila,
    CAST(floor(longitude / {GRADO}) AS INT) AS col,
    year(acq_date)                          AS año,
    month(acq_date)                         AS mes,
    scan * track                            AS area_px,     -- km² del píxel
    TRY_CAST(confidence AS INT)             AS confianza,   -- 0–100 en MODIS (VIIRS usa letras)
    ((acq_time // 100) + (acq_time % 100) / 60 + longitude / 15 + 48) % 24 AS hora_local
FROM nasa.hotspots
""")

**Paso 2 · Meses buenos de cada satélite.** Un mes sirve si el satélite tiene datos todos los días y ninguno con falla. Un día tiene falla si el satélite detectó menos del 25 % de lo normal para él, medido con la mediana de los 45 días antes y después. Ejemplo: Aqua del 16 de agosto al 2 de septiembre de 2020. Pasa en 37 de 22,690 días.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW meses AS
WITH por_dia AS (
    SELECT satellite, año, mes, acq_date, count(*) AS n
    FROM detecciones
    GROUP BY ALL
),
con_falla AS (
    SELECT *,
           n < 0.25 * median(n) OVER (PARTITION BY satellite ORDER BY acq_date
                                      RANGE BETWEEN INTERVAL 45 DAYS PRECEDING
                                                AND INTERVAL 45 DAYS FOLLOWING) AS falla
    FROM por_dia
),
por_satelite AS (
    SELECT satellite, año, mes,
           count(*) = day(last_day(make_date(año, mes, 1))) AND NOT bool_or(falla) AS bueno
    FROM con_falla
    GROUP BY satellite, año, mes
)
SELECT año, mes,
       bool_or(satellite = 'Terra' AND bueno) AS terra_ok,
       bool_or(satellite = 'Aqua'  AND bueno) AS aqua_ok,
       CASE WHEN bool_or(satellite = 'N'   AND bueno) THEN 'N'
            WHEN bool_or(satellite = 'N20' AND bueno) THEN 'N20' END AS viirs_sat   -- el VIIRS activo
FROM por_satelite
GROUP BY año, mes
""")

**Paso 3 · Hora de paso de Terra y Aqua cada mes.** Es la mediana de la hora local de sus detecciones de día cerca del ecuador. Muestra cómo se corren las órbitas desde 2022.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW paso AS
SELECT año, mes,
       median(hora_local) FILTER (satellite = 'Terra') AS hora_paso_terra,
       median(hora_local) FILTER (satellite = 'Aqua')  AS hora_paso_aqua
FROM detecciones
WHERE sensor = 'MODIS' AND daynight = 'D' AND abs(latitude) < 30
GROUP BY año, mes
""")

**Paso 4 · MODIS por celda y mes (las entradas).**

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW modis AS
SELECT fila, col, año, mes,
       count(*) FILTER (satellite = 'Terra' AND daynight = 'D') AS terra_dia,
       count(*) FILTER (satellite = 'Terra' AND daynight = 'N') AS terra_noche,
       count(*) FILTER (satellite = 'Aqua'  AND daynight = 'D') AS aqua_dia,
       count(*) FILTER (satellite = 'Aqua'  AND daynight = 'N') AS aqua_noche,
       coalesce(sum(frp) FILTER (satellite = 'Terra'), 0)       AS terra_frp,
       coalesce(sum(frp) FILTER (satellite = 'Aqua'), 0)        AS aqua_frp,
       max(frp)                                 AS modis_frp_max,
       avg(area_px)                             AS modis_area_px,         -- crece hacia el borde de la pasada
       avg(confianza)                           AS modis_confianza,
       avg(CAST(confianza < 30 AS INT))         AS modis_frac_baja_conf,
       count(DISTINCT acq_date)                 AS modis_dias,            -- días con fuego en el mes
       avg(CAST(type = 2 AS INT))               AS modis_frac_fijas       -- industria y otras fuentes fijas
FROM detecciones
WHERE sensor = 'MODIS'
GROUP BY fila, col, año, mes
""")

**Paso 5 · Fuego en las 8 celdas vecinas.** Cada celda le pasa su conteo a sus 8 vecinas y luego cada celda suma lo que recibió.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW vecinos AS
SELECT m.fila + df AS fila,
       m.col + dc  AS col,
       m.año, m.mes,
       sum(m.terra_dia + m.terra_noche + m.aqua_dia + m.aqua_noche) AS modis_vecinos
FROM modis m
CROSS JOIN (SELECT unnest([-1, 0, 1]) AS df)
CROSS JOIN (SELECT unnest([-1, 0, 1]) AS dc)
WHERE NOT (df = 0 AND dc = 0)   -- sin contar la celda misma
GROUP BY ALL
""")

**Paso 6 · VIIRS por celda y mes (lo que se quiere predecir).**

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW viirs AS
SELECT fila, col, año, mes, satellite,
       count(*) AS viirs_n,
       sum(frp) AS viirs_frp
FROM detecciones
WHERE sensor LIKE 'VIIRS%'
GROUP BY fila, col, año, mes, satellite
""")

**Paso 7 · Unir todo.** Hay una fila por cada celda que alguna vez tuvo fuego y por cada mes. Se incluyen los meses sin detecciones, porque si no el modelo nunca aprendería a predecir cero.

Se agregan tres columnas más: el fuego del mes anterior, el del mes siguiente y el promedio histórico de MODIS en la celda para ese mes del año.

El resultado se guarda en un archivo para no recalcular: la primera vez tarda ~1 min. **Si cambias algo del SQL, borra ese archivo** para que se vuelva a armar.

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW tabla_celdas AS
WITH celdas AS (SELECT DISTINCT fila, col FROM detecciones),
rejilla AS (   -- todas las celdas × todos los meses, con ceros donde no hubo detecciones
    SELECT c.fila, c.col, m.*, p.hora_paso_terra, p.hora_paso_aqua,
           coalesce(md.terra_dia, 0)            AS terra_dia,
           coalesce(md.terra_noche, 0)          AS terra_noche,
           coalesce(md.aqua_dia, 0)             AS aqua_dia,
           coalesce(md.aqua_noche, 0)           AS aqua_noche,
           coalesce(md.terra_dia + md.terra_noche + md.aqua_dia + md.aqua_noche, 0) AS modis_n,
           coalesce(md.terra_frp, 0)            AS terra_frp,
           coalesce(md.aqua_frp, 0)             AS aqua_frp,
           coalesce(md.modis_frp_max, 0)        AS modis_frp_max,
           md.modis_area_px,                    -- sin detecciones no hay píxel ni confianza: queda vacío
           md.modis_confianza,
           coalesce(md.modis_frac_baja_conf, 0) AS modis_frac_baja_conf,
           coalesce(md.modis_dias, 0)           AS modis_dias,
           coalesce(md.modis_frac_fijas, 0)     AS modis_frac_fijas,
           coalesce(vc.modis_vecinos, 0)        AS modis_vecinos,
           -- VIIRS solo en sus meses buenos; en los demás queda vacío (no se sabe)
           CASE WHEN m.viirs_sat IS NOT NULL THEN coalesce(v.viirs_n, 0) END   AS viirs_n,
           CASE WHEN m.viirs_sat IS NOT NULL THEN coalesce(v.viirs_frp, 0) END AS viirs_frp
    FROM celdas c
    CROSS JOIN meses m
    LEFT JOIN paso p     ON (p.año, p.mes) = (m.año, m.mes)
    LEFT JOIN modis md   ON (md.fila, md.col, md.año, md.mes) = (c.fila, c.col, m.año, m.mes)
    LEFT JOIN vecinos vc ON (vc.fila, vc.col, vc.año, vc.mes) = (c.fila, c.col, m.año, m.mes)
    LEFT JOIN viirs v    ON (v.fila, v.col, v.año, v.mes, v.satellite) = (c.fila, c.col, m.año, m.mes, m.viirs_sat)
)
SELECT *,
       ({GRADO} * (fila + 0.5))::FLOAT AS lat,   -- centro de la celda
       ({GRADO} * (col + 0.5))::FLOAT  AS lon,
       lag(modis_n)  OVER celda AS modis_mes_anterior,
       lead(modis_n) OVER celda AS modis_mes_siguiente,
       avg(CASE WHEN año BETWEEN 2003 AND 2021 AND terra_ok AND aqua_ok THEN modis_n END)
           OVER (PARTITION BY fila, col, mes) AS modis_historico   -- años con órbitas normales
FROM rejilla
WINDOW celda AS (PARTITION BY fila, col ORDER BY año, mes)
""")

if not ARCHIVO_TABLA.exists():
    ARCHIVO_TABLA.parent.mkdir(exist_ok=True)
    con.execute(f"COPY (SELECT * FROM tabla_celdas ORDER BY año, mes, fila, col) TO '{ARCHIVO_TABLA}'")

tabla = pd.read_parquet(ARCHIVO_TABLA)
numericas = tabla.columns.difference(["fila", "col", "año", "mes", "terra_ok", "aqua_ok", "viirs_sat"])
tabla[numericas] = tabla[numericas].astype("float32")  # la mitad de memoria
print(f"{len(tabla):,} filas = {tabla[['fila', 'col']].drop_duplicates().shape[0]:,} celdas "
      f"× {tabla[['año', 'mes']].drop_duplicates().shape[0]} meses")

---
## 3. Variables calculadas (feature engineering)
Son columnas nuevas calculadas a partir de las que ya hay:

| Variable | Qué es | Por qué podría ayudar |
|---|---|---|
| `horas_de_sol` | duración del día en la celda | el fuego tiene un ciclo diario |
| `cenit_terra`, `cenit_aqua` | ángulo del Sol respecto a la vertical a la hora en que pasa el satélite (0° = Sol encima) | con el Sol alto, el suelo está más caliente y cuesta más distinguir un fuego |
| `distancia_sol` | distancia Tierra–Sol (0.983 en enero, 1.017 en julio) | se pidió probarla; solo depende de la fecha |
| `frp_por_det_terra`, `frp_por_det_aqua` | FRP promedio de cada detección | los fuegos chicos son los que VIIRS ve y MODIS no |
| `razon_aqua_terra` | log(Aqua ÷ Terra) | si es alta, el fuego se concentra en la tarde, que es cuando pasa VIIRS |
| `frac_noche` | % de detecciones MODIS de noche | de noche VIIRS ve ~26 veces más que Aqua; de día, ~6 veces |

**Por qué el ángulo del Sol usa la hora nominal y no la real:** primero se probó con la hora real de paso de cada mes (paso 3). Esa hora casi no cambia en los años de entrenamiento, pero desde 2022 se sale del rango: Aqua llega a pasar a las 14:46. La red neuronal nunca vio esas horas, extrapoló sin control y se equivocó un **58 %** en 2022–2025. Con la hora nominal (10:30 para Terra y 13:30 para Aqua), el ángulo solo depende del lugar y del mes, y el modelo queda estable.

In [ ]:
DIA_DEL_AÑO = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])  # el día 15 de cada mes


HORA_NOMINAL = {"terra": 10.5, "aqua": 13.5}  # hora solar local de diseño de cada órbita


def variables_sol(t):
    """Posición del Sol a mediados del mes (fórmulas astronómicas estándar)."""
    dia = DIA_DEL_AÑO[t.mes - 1]
    lat = np.radians(t.lat)
    declinacion = np.radians(23.44) * np.sin(2 * np.pi * (284 + dia) / 365)  # latitud donde el Sol cae vertical
    t["distancia_sol"] = 1 - 0.01672 * np.cos(2 * np.pi * (dia - 4) / 365.25)
    t["horas_de_sol"] = 24 / np.pi * np.arccos(np.clip(-np.tan(lat) * np.tan(declinacion), -1, 1))
    for sat in ("terra", "aqua"):
        angulo_horario = np.radians(15 * (HORA_NOMINAL[sat] - 12))  # 15° por cada hora desde el mediodía
        cos_cenit = np.sin(lat) * np.sin(declinacion) + np.cos(lat) * np.cos(declinacion) * np.cos(angulo_horario)
        t[f"cenit_{sat}"] = np.degrees(np.arccos(np.clip(cos_cenit, -1, 1)))
    return t


def variables_intensidad(t):
    """Qué tan grandes son los fuegos y a qué hora del día queman."""
    terra = t.terra_dia + t.terra_noche
    aqua = t.aqua_dia + t.aqua_noche
    t["frp_por_det_terra"] = (t.terra_frp / terra).where(terra > 0, 0)
    t["frp_por_det_aqua"] = (t.aqua_frp / aqua).where(aqua > 0, 0)
    t["razon_aqua_terra"] = np.log1p(aqua) - np.log1p(terra)
    t["frac_noche"] = ((t.terra_noche + t.aqua_noche) / t.modis_n).where(t.modis_n > 0, 0)
    return t


tabla = variables_intensidad(variables_sol(tabla))
nuevas = ["horas_de_sol", "cenit_terra", "cenit_aqua", "distancia_sol",
          "frp_por_det_terra", "frp_por_det_aqua", "razon_aqua_terra", "frac_noche"]
tabla[nuevas] = tabla[nuevas].astype("float32")
tabla[nuevas].describe().round(2)

Todas las entradas del modelo, agrupadas:

In [ ]:
GRUPOS = {
    "MODIS": ["terra_dia", "terra_noche", "aqua_dia", "aqua_noche", "terra_frp", "aqua_frp",
              "modis_frp_max", "modis_area_px", "modis_confianza", "modis_frac_baja_conf",
              "modis_dias", "modis_frac_fijas"],
    "intensidad": ["frp_por_det_terra", "frp_por_det_aqua", "razon_aqua_terra", "frac_noche"],
    "vecindad y tiempo": ["modis_vecinos", "modis_mes_anterior", "modis_mes_siguiente", "modis_historico"],
    "sol": ["cenit_terra", "cenit_aqua", "horas_de_sol", "distancia_sol"],
    "lugar y fecha": ["lat", "lon", "mes"],
}
ENTRADAS = [entrada for grupo in GRUPOS.values() for entrada in grupo]
print(len(ENTRADAS), "entradas")

---
## 4. Separar años
Se separan años completos, no filas al azar. Los meses vecinos se parecen mucho, y si se mezclaran el modelo parecería mejor de lo que es.

| Conjunto | Años | Para qué |
|---|---|---|
| entrenamiento | 2012–2016, 2018–2020 | aprender (órbitas de MODIS normales) |
| prueba | 2017 (S-NPP), 2021 (NOAA-20) | medir qué tan bien predice en años que no vio |
| órbitas corridas | 2022–2025 | ver si el modelo aguanta la deriva (no se usa para aprender) |
| solo MODIS | 2002–2011 | aplicar el modelo |

In [ ]:
AÑOS_ENTRENAMIENTO = [2012, 2013, 2014, 2015, 2016, 2018, 2019, 2020]
AÑOS_PRUEBA = [2017, 2021]

modis_bueno = tabla.terra_ok & tabla.aqua_ok
con_viirs = modis_bueno & tabla.viirs_sat.notna()

entrenamiento = tabla[con_viirs & tabla.año.isin(AÑOS_ENTRENAMIENTO)]
prueba = tabla[con_viirs & tabla.año.isin(AÑOS_PRUEBA)]
todo_viirs = tabla[con_viirs]  # 2012–2025: para ver la deriva
solo_modis = tabla[modis_bueno]  # 2002–2025: para la serie armonizada

print(f"entrenamiento: {len(entrenamiento):,} filas · prueba: {len(prueba):,} filas")

---
## 5. Tres modelos
Los tres se usan igual:
- `modelo.entrenar(datos)` aprende.
- `modelo.predecir(datos)` devuelve una tabla con `detecciones` y `frp` para cada fila.

Con `predecir(datos, satelite="N20")` se elige qué VIIRS imitar: `"N"` para S-NPP o `"N20"` para NOAA-20. Si no se indica, usa el que estaba activo ese mes.

### 5.1 Referencia
La forma más simple: multiplicar MODIS por la razón VIIRS/MODIS promedio de su región y su mes, como vimos en el EDA. Cualquier modelo tiene que ganarle a esta. Su punto débil es que, si MODIS no vio nada en la celda, predice cero aunque VIIRS sí haya visto fuegos pequeños.

In [ ]:
def region_10_grados(datos):
    """Bloque de 10° × 10° al que pertenece cada celda."""
    return ((datos.lat + 90) // 10 * 36 + (datos.lon + 180) // 10).astype(int)


def razones(sumas):
    """Razón VIIRS / MODIS para detecciones y FRP (vacía si MODIS no vio nada)."""
    return pd.DataFrame({
        "detecciones": sumas.viirs_n / sumas.modis_n,
        "frp": sumas.viirs_frp / sumas.modis_frp,
    }).replace(np.inf, np.nan)


class Referencia:
    """VIIRS = MODIS × razón VIIRS/MODIS de su región (10° × 10°) en ese mes del año.
    Si la región no tuvo fuego al entrenar, usa la razón de todo el mundo en ese mes."""

    def entrenar(self, datos):
        sumas = datos[["mes", "modis_n", "viirs_n", "viirs_frp"]].assign(
            region=region_10_grados(datos), modis_frp=datos.terra_frp + datos.aqua_frp)
        self.razon_region = razones(sumas.groupby(["region", "mes"]).sum())
        self.razon_mes = razones(sumas.drop(columns="region").groupby("mes").sum())
        return self

    def predecir(self, datos, satelite=None):  # no distingue entre S-NPP y NOAA-20
        llave = pd.MultiIndex.from_arrays([region_10_grados(datos), datos.mes])
        razon = self.razon_region.reindex(llave).to_numpy()
        respaldo = self.razon_mes.reindex(datos.mes).to_numpy()
        razon = np.where(np.isnan(razon), respaldo, razon)
        modis = np.column_stack([datos.modis_n, datos.terra_frp + datos.aqua_frp])
        return pd.DataFrame(modis * razon, columns=["detecciones", "frp"], index=datos.index)


referencia = Referencia().entrenar(entrenamiento)

### 5.2 LightGBM
Son árboles de decisión en serie, donde cada árbol corrige el error de los anteriores. Suele ser el método más fuerte con datos en tabla.

Se entrenan dos modelos:
- **Detecciones,** con pérdida *Poisson*, hecha para conteos.
- **FRP,** con pérdida *Tweedie*, para valores con muchos ceros.

Tarda ~3 min.

In [ ]:
class ModeloLightGBM:
    def __init__(self, arboles=600, paso=0.05):
        """Más árboles con pasos más cortos = más preciso pero más lento."""
        self.parametros = dict(n_estimators=arboles, learning_rate=paso, num_leaves=127, min_child_samples=100,
                               subsample=0.5, subsample_freq=1, colsample_bytree=0.9, verbose=-1)

    def entradas(self, datos, satelite=None):
        """Las entradas + una columna que dice qué VIIRS imitar (0 = S-NPP, 1 = NOAA-20)."""
        satelite = datos.viirs_sat if satelite is None else satelite
        return datos[ENTRADAS].assign(noaa20=np.where(satelite == "N20", 1, 0))

    def entrenar(self, datos):
        x = self.entradas(datos)
        self.detecciones = lgb.LGBMRegressor(objective="poisson", **self.parametros).fit(x, datos.viirs_n)
        self.frp = lgb.LGBMRegressor(objective="tweedie", **self.parametros).fit(x, datos.viirs_frp)
        return self

    def predecir(self, datos, satelite=None):
        x = self.entradas(datos, satelite)
        return pd.DataFrame({"detecciones": self.detecciones.predict(x), "frp": self.frp.predict(x)},
                            index=datos.index)

    def importancia(self):
        """% de la mejora total de los árboles que aporta cada entrada."""
        tabla = pd.DataFrame({"detecciones": self.detecciones.booster_.feature_importance("gain"),
                              "FRP": self.frp.booster_.feature_importance("gain")},
                             index=self.detecciones.feature_name_)
        return (100 * tabla / tabla.sum()).round(1)


lightgbm = ModeloLightGBM().entrenar(entrenamiento)

### 5.3 Red neuronal
Entran las mismas variables y salen 4 valores: detecciones y FRP de S-NPP, y detecciones y FRP de NOAA-20.

```
27 entradas → 256 → 128 → 64 neuronas → [det S-NPP, det NOAA-20, FRP S-NPP, FRP NOAA-20]
```

En estos datos S-NPP y NOAA-20 nunca coinciden en el tiempo. Por eso, en cada fila solo se corrigen las 2 salidas del satélite que estaba activo ese mes. Tarda ~1 min.

In [ ]:
def error_poisson(log_prediccion, real, usar):
    """Error de Poisson (devianza / 2) promedio, solo en las salidas observadas. 0 = perfecto."""
    error = torch.exp(log_prediccion) - real * log_prediccion + torch.special.xlogy(real, real) - real
    return (error * usar).sum() / usar.sum()


class RedNeuronal:
    SALIDAS = ["det S-NPP", "det NOAA-20", "FRP S-NPP", "FRP NOAA-20"]
    ESCALA_FRP = 100  # el FRP se aprende en cientos de MW, para quedar en una escala parecida a los conteos
    EN_LOG = ["terra_dia", "terra_noche", "aqua_dia", "aqua_noche", "terra_frp", "aqua_frp",
              "modis_frp_max", "modis_dias", "frp_por_det_terra", "frp_por_det_aqua",
              "modis_vecinos", "modis_mes_anterior", "modis_mes_siguiente", "modis_historico"]

    def preparar(self, datos, ajustar=False):
        """Entradas listas para la red:
        - conteos y FRP en log, para que los números queden de tamaño parecido
        - mes y longitud como ángulos (seno y coseno), para que diciembre quede junto a enero
        - todo estandarizado (media 0, desviación 1) con la media y desviación del entrenamiento"""
        x = datos[ENTRADAS].fillna(0)
        x = x.assign(**{c: np.log1p(x[c].clip(lower=0)) for c in self.EN_LOG},
                     mes_sen=np.sin(2 * np.pi * x.mes / 12), mes_cos=np.cos(2 * np.pi * x.mes / 12),
                     lon_sen=np.sin(np.radians(x.lon)), lon_cos=np.cos(np.radians(x.lon)))
        x = x.drop(columns=["mes", "lon"]).to_numpy("float32")
        if ajustar:
            self.media, self.desviacion = x.mean(axis=0), x.std(axis=0) + 1e-6
        return torch.tensor((x - self.media) / self.desviacion)

    def objetivos(self, datos):
        """Lo que la red debe aprender, y qué salidas se observaron (las del VIIRS activo)."""
        es_noaa20 = datos.viirs_sat == "N20"
        y = pd.DataFrame({
            "det S-NPP": datos.viirs_n.where(~es_noaa20),
            "det NOAA-20": datos.viirs_n.where(es_noaa20),
            "FRP S-NPP": datos.viirs_frp.where(~es_noaa20) / self.ESCALA_FRP,
            "FRP NOAA-20": datos.viirs_frp.where(es_noaa20) / self.ESCALA_FRP,
        })
        return torch.tensor(y.fillna(0).to_numpy("float32")), torch.tensor(y.notna().to_numpy("float32"))

    def entrenar(self, datos, epocas=10, lote=8192):
        x = self.preparar(datos, ajustar=True)
        y, usar = self.objetivos(datos)
        self.red = nn.Sequential(
            nn.Linear(x.shape[1], 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 4),  # salen en log: así la predicción siempre es positiva
        )
        with torch.no_grad():  # arranca prediciendo el promedio de cada salida
            self.red[-1].bias.copy_(torch.log((y * usar).sum(axis=0) / usar.sum(axis=0)))
        optimizador = torch.optim.Adam(self.red.parameters(), lr=1e-3)
        ritmo = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=epocas)  # pasos cada vez más cortos

        for epoca in range(1, epocas + 1):
            error_total = 0.0
            for filas in torch.randperm(len(x)).split(lote):  # lotes al azar
                error = error_poisson(self.red(x[filas]), y[filas], usar[filas])
                optimizador.zero_grad()
                error.backward()
                nn.utils.clip_grad_norm_(self.red.parameters(), 1.0)
                optimizador.step()
                error_total += error.item() * len(filas)
            ritmo.step()
            print(f"época {epoca:2d}/{epocas} · error {error_total / len(x):.2f}")
        return self

    def predecir(self, datos, satelite=None, bloque=1_000_000):
        with torch.no_grad():  # por bloques, para no llenar la memoria
            partes = [torch.exp(self.red(self.preparar(datos.iloc[i:i + bloque]))).numpy()
                      for i in range(0, len(datos), bloque)]
        salidas = pd.DataFrame(np.vstack(partes), columns=self.SALIDAS, index=datos.index)
        satelite = datos.viirs_sat if satelite is None else satelite
        es_noaa20 = np.asarray(satelite == "N20")
        return pd.DataFrame({
            "detecciones": np.where(es_noaa20, salidas["det NOAA-20"], salidas["det S-NPP"]),
            "frp": np.where(es_noaa20, salidas["FRP NOAA-20"], salidas["FRP S-NPP"]) * self.ESCALA_FRP,
        }, index=datos.index)


red = RedNeuronal().entrenar(entrenamiento)

---
## 6. ¿Qué tan bien predicen?
Se miden en los años de prueba, que ningún modelo vio al entrenar:
- **Error en el total:** cuánto se pasa (+) o se queda corto (−) al sumar todo el mundo y todos los meses. 0 % es perfecto.
- **Error mensual típico:** en un mes cualquiera, cuánto se equivoca en el total global. Es la mediana de los meses.
- **R² por celda:** qué tanto acierta celda por celda, en escala log. 1 es perfecto; 0 significa que no es mejor que adivinar el promedio.

In [ ]:
def totales_mensuales(datos, predicciones):
    """Suma de todo el mundo en cada mes: lo observado y lo que predice cada modelo."""
    columnas = {("detecciones", "Observado"): datos.viirs_n, ("frp", "Observado"): datos.viirs_frp}
    for nombre, prediccion in predicciones.items():
        columnas[("detecciones", nombre)] = prediccion.detecciones
        columnas[("frp", nombre)] = prediccion.frp
    return pd.DataFrame(columnas).groupby([datos.año, datos.mes]).sum()


def evaluar(datos, predicciones):
    totales = totales_mensuales(datos, predicciones)
    filas = []
    for variable, observado in (("detecciones", datos.viirs_n), ("frp", datos.viirs_frp)):
        for nombre, prediccion in predicciones.items():
            mensual = totales[(variable, nombre)] / totales[(variable, "Observado")] - 1
            filas.append({
                "variable": variable,
                "modelo": nombre,
                "error en el total (%)": 100 * (prediccion[variable].sum() / observado.sum() - 1),
                "error mensual típico (%)": 100 * mensual.abs().median(),
                "R² por celda": r2_score(np.log1p(observado), np.log1p(prediccion[variable].clip(lower=0))),
            })
    return pd.DataFrame(filas).set_index(["variable", "modelo"]).round(3)

In [ ]:
modelos = {"Referencia": referencia, "LightGBM": lightgbm, "Red neuronal": red}
predicciones = {nombre: modelo.predecir(prueba) for nombre, modelo in modelos.items()}

display(evaluar(prueba, predicciones))
graf.prueba(totales_mensuales(prueba, predicciones), satelites={2017: "VIIRS S-NPP", 2021: "VIIRS NOAA-20"})

---
## 7. ¿Qué entradas usa el modelo?
Primero se probó cada grupo de variables nuevas por separado: se agregaba un grupo a la vez y se medía el resultado con una versión rápida de LightGBM (300 árboles). Estos fueron los resultados en los años de prueba:

| Entradas | R² detecciones | Error mensual detecciones | R² FRP | Error mensual FRP |
|---|---|---|---|---|
| las 15 originales (MODIS, lugar y fecha) | 0.812 | 1.62 % | 0.763 | 1.75 % |
| + sol y hora de paso real | 0.813 | 1.69 % | 0.763 | 1.42 % |
| + intensidad | 0.812 | 1.63 % | 0.764 | 1.56 % |
| + vecindad | 0.818 | 1.48 % | 0.775 | 1.48 % |
| + mes anterior y siguiente | 0.819 | 1.49 % | 0.774 | 1.79 % |
| **todas** | **0.824** | **1.29 %** | **0.783** | **1.18 %** |

Ningún grupo mejora mucho por sí solo, pero juntos bajan el error mensual entre un 20 % y un 30 %. La gráfica muestra cuánto usa el modelo final cada entrada.

In [ ]:
graf.importancia(lightgbm.importancia(), GRUPOS)

---
## 8. ¿Qué pasa con las órbitas corridas de MODIS?
Desde 2022, Terra pasa cada vez más temprano y Aqua cada vez más tarde. En 2025, Terra pasa a las 9:25 en vez de las 10:30, y Aqua a las 14:46 en vez de las 13:30.

Los años 2022–2025 no se usaron para entrenar, y el modelo no sabe a qué hora pasó realmente cada satélite. Si el error crece en esos años, el modelo solo es confiable con órbitas normales, como las de 2002–2011, que es donde se aplica.

In [ ]:
def a_fecha(indice):
    """Índice (año, mes) → fecha del primer día del mes."""
    return pd.to_datetime([f"{año}-{mes:02d}-01" for año, mes in indice])


def errores_mensuales(datos, modelos):
    """Error (%) del total global de cada mes, para cada modelo. + = predice de más."""
    totales = totales_mensuales(datos, {nombre: modelo.predecir(datos) for nombre, modelo in modelos.items()})
    errores = pd.DataFrame({
        (variable, nombre): 100 * (totales[(variable, nombre)] / totales[(variable, "Observado")] - 1)
        for variable in ("detecciones", "frp") for nombre in modelos
    })
    errores.index = a_fecha(errores.index)
    return errores


errores = errores_mensuales(todo_viirs, modelos)
graf.deriva(errores, AÑOS_PRUEBA)

periodo = np.select([errores.index.year >= 2022, errores.index.year.isin(AÑOS_PRUEBA)],
                    ["órbitas corridas (2022–2025)", "años de prueba"], "años de entrenamiento")
errores.abs().groupby(periodo).median().round(1).rename_axis("error mensual típico (%)")

---
## 9. Validación año por año
Hasta aquí el error se midió solo en 2 años de prueba. Para medirlo en los 10 años de 2012–2021, se entrena LightGBM 10 veces, dejando fuera un año distinto cada vez, y se predice el año que quedó fuera. Así cada mes de 2012–2021 tiene una predicción hecha sin haberlo visto.

Esta validación da dos cosas:
- **Dónde se equivoca:** el mapa de la sección 10.
- **Cuánto se puede equivocar en un mes:** el rango de incertidumbre de la sección 11.

Se usa una versión rápida de LightGBM (300 árboles con pasos más largos), que en el experimento de la sección 7 dio casi la misma precisión. La primera vez tarda ~12 min; el resultado se guarda en `data/modelo/`.

In [ ]:
ARCHIVO_VALIDACION = ROOT / "data" / "modelo" / "validacion_por_año.parquet"
AÑOS_VALIDACION = range(2012, 2022)

if not ARCHIVO_VALIDACION.exists():
    datos = tabla[con_viirs & tabla.año.isin(AÑOS_VALIDACION)]
    partes = []
    for año in AÑOS_VALIDACION:
        modelo = ModeloLightGBM(arboles=300, paso=0.1).entrenar(datos[datos.año != año])
        fuera = datos[datos.año == año]
        partes.append(fuera[["lat", "lon", "año", "mes", "viirs_n", "viirs_frp"]].join(modelo.predecir(fuera)))
        print(año, "listo")
    pd.concat(partes).to_parquet(ARCHIVO_VALIDACION)

validacion = pd.read_parquet(ARCHIVO_VALIDACION)
evaluar(validacion, {"LightGBM (un año fuera a la vez)": validacion[["detecciones", "frp"]]})

---
## 10. ¿Dónde se equivoca?
Se agrupan las celdas en bloques de 2° (~220 km) y se mide, con las predicciones de la validación:
- **Sesgo:** si en el total de 2012–2021 el modelo se pasa (rojo) o se queda corto (azul) en ese bloque.
- **Error típico de un mes:** qué tanto se equivoca en un mes cualquiera en ese bloque. Solo cuenta meses con al menos 10 detecciones, porque con pocas detecciones los porcentajes engañan.

Los bloques con menos de 1,000 detecciones VIIRS en 10 años quedan en gris.

In [ ]:
def error_por_bloque(validacion, grado=2, minimo=1000):
    v = validacion.assign(bloque_lat=validacion.lat // grado * grado, bloque_lon=validacion.lon // grado * grado)
    por_mes = v.groupby(["bloque_lat", "bloque_lon", "año", "mes"])[["viirs_n", "detecciones"]].sum()
    por_mes["error"] = 100 * (por_mes.detecciones / por_mes.viirs_n - 1)

    bloques = por_mes.groupby(level=["bloque_lat", "bloque_lon"]).agg(
        observado=("viirs_n", "sum"), estimado=("detecciones", "sum"))
    bloques["sesgo"] = 100 * (bloques.estimado / bloques.observado - 1)
    bloques["error_mensual"] = por_mes[por_mes.viirs_n >= 10].error.abs().groupby(
        level=["bloque_lat", "bloque_lon"]).median()
    bloques["confiable"] = bloques.observado >= minimo
    return bloques.reset_index()


bloques = error_por_bloque(validacion)
graf.mapa_error(bloques, grado=2)

# ¿Qué parte del fuego del mundo está en bloques donde el modelo acierta bien?
medidos = bloques[bloques.confiable]
for limite in (10, 20):
    parte = medidos.observado[medidos.sesgo.abs() <= limite].sum() / bloques.observado.sum()
    print(f"{100 * parte:.0f} % de las detecciones VIIRS están en bloques con sesgo menor a ±{limite} %")

---
## 11. Serie armonizada 2002–2025
Este es el resultado final: LightGBM aplicado a cada celda y cada mes en que MODIS tiene datos completos, en unidades de VIIRS NOAA-20.
- **2012–2025:** se puede comparar con lo que VIIRS realmente observó.
- **Antes de 2012:** es lo que VIIRS habría visto.

**Rango de incertidumbre:** sale de la validación. Es el rango en que cayó el error del total mundial en el 90 % de los meses de 2012–2021. Para una región chica el rango es más ancho; el mapa anterior muestra dónde.

Se guardan dos archivos en `data/modelo/`:
- `armonizado_celdas.parquet`: cada celda y mes. Es la base para el calendario de quemas por zona.
- `serie_armonizada.csv`: el total mundial de cada mes.

In [ ]:
error_mensual = validacion.groupby(["año", "mes"])[["viirs_n", "viirs_frp", "detecciones", "frp"]].sum()
RANGO = {
    "detecciones": (error_mensual.detecciones / error_mensual.viirs_n - 1).quantile([0.05, 0.95]).to_numpy(),
    "frp": (error_mensual.frp / error_mensual.viirs_frp - 1).quantile([0.05, 0.95]).to_numpy(),
}
for variable, (bajo, alto) in RANGO.items():
    print(f"{variable}: en el 90 % de los meses el error del total mundial estuvo entre {100 * bajo:+.1f} % y {100 * alto:+.1f} %")

In [ ]:
# Cada celda y mes, en unidades de VIIRS NOAA-20 (~2 min)
celdas_armonizadas = solo_modis[["fila", "col", "lat", "lon", "año", "mes"]].join(
    lightgbm.predecir(solo_modis, satelite="N20"))
celdas_armonizadas.to_parquet(ROOT / "data" / "modelo" / "armonizado_celdas.parquet")


def serie_armonizada(celdas):
    """Total mundial de cada mes: MODIS tal cual, MODIS armonizado (con su rango del 90 %) y VIIRS observado."""
    modis = solo_modis.assign(frp=solo_modis.terra_frp + solo_modis.aqua_frp).groupby(["año", "mes"]).agg(
        detecciones=("modis_n", "sum"), frp=("frp", "sum"))
    estimado = celdas.groupby(["año", "mes"])[["detecciones", "frp"]].sum()
    observado = tabla[tabla.viirs_sat.notna()].groupby(["año", "mes"]).agg(
        detecciones=("viirs_n", "sum"), frp=("viirs_frp", "sum"), satelite=("viirs_sat", "first"))
    serie = pd.concat([modis.add_suffix("_modis"), estimado.add_suffix("_estimado"),
                       observado.add_suffix("_observado")], axis=1).sort_index()
    for variable, (bajo, alto) in RANGO.items():  # si el modelo se pasa un X %, lo real es estimado / (1 + X)
        serie[f"{variable}_min"] = serie[f"{variable}_estimado"] / (1 + alto)
        serie[f"{variable}_max"] = serie[f"{variable}_estimado"] / (1 + bajo)
    serie.index = a_fecha(serie.index)
    return serie


serie = serie_armonizada(celdas_armonizadas)
serie.to_csv(ROOT / "data" / "modelo" / "serie_armonizada.csv", index_label="mes")
graf.serie(serie)

---
## 12. Antes y después de armonizar
La gráfica resume para qué sirve todo esto. Si se unen los registros tal cual, MODIS hasta 2011 y VIIRS desde 2012, en las detecciones aparece un salto falso en 2012: VIIRS ve unas 4.6 veces más, pero no hubo más fuego, solo cambió el sensor. Armonizado, el registro queda continuo.

En el FRP total casi no hay salto (1.05×). Los dos sensores miden una energía total parecida; VIIRS solo la reparte en más píxeles, que son más pequeños.

Cada punto es el promedio mensual de un año. Solo se usan años con al menos 9 meses buenos.

In [ ]:
buenos = serie.notna().groupby(serie.index.year).sum()
anual = serie.groupby(serie.index.year).mean(numeric_only=True)
for columna in anual.columns:
    anual.loc[buenos[columna] < 9, columna] = np.nan

coinciden = anual.loc[2012:2021]  # años con MODIS y VIIRS a la vez
factor = {v: (coinciden[f"{v}_observado"] / coinciden[f"{v}_modis"]).mean() for v in ("detecciones", "frp")}
graf.antes_y_despues(anual, factor)